In [ ]:
!pip install fomo-edge-ai==0.0.12

You must restart the session after running the above

In [ ]:

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
!wget https://openwear.ai/cdn/dog-no-augmentation.yolov8.zip

In [ ]:
!unzip dog-no-augmentation.yolov8.zip

In [ ]:
import shutil
import zipfile
from pathlib import Path
import yaml
from huggingface_hub import hf_hub_download

DATASET_ROOT = Path("dog-no-augmentation.yolov8")
data_yaml_path = DATASET_ROOT / "data.yaml"

# Quick sanity check — count images and labels
def count_split(split_dir):
    imgs = list(split_dir.rglob("*.jpg")) + list(split_dir.rglob("*.png")) + list(split_dir.rglob("*.jpeg"))
    lbl_dir = split_dir.parent / "labels"
    lbls = list(lbl_dir.rglob("*.txt")) if lbl_dir.exists() else []
    return len(imgs), len(lbls)

train_imgs, train_lbls = count_split(DATASET_ROOT / "train" / "images")
val_imgs, val_lbls     = count_split(DATASET_ROOT / "valid" / "images")
test_imgs, test_lbls   = count_split(DATASET_ROOT / "test" / "images")
print(f"Combined Train : {train_imgs} images, {train_lbls} label files")
print(f"Combined Val   : {val_imgs} images, {val_lbls} label files")
print(f"Combined Test  : {test_imgs} images, {test_lbls} label files")


## 3 — Train FOMO-m

FOMO-m uses a **MobileNetV2 backbone (α=0.5)** with a single-pixel detection head.
Input resolution is **192** → output grid **24x24** (8× downsample).

Training uses:
- Adam, lr=3e-4, ReduceLROnPlateau on val F1
- Weighted CrossEntropy (background=1×, foreground=100×)
- No augmentation (plain resize + ImageNet normalisation)
- Val sweep over confidence thresholds every epoch

In [ ]:
from fomo import FOMO

EPOCHS     = 100
BATCH      = 16
MODEL_SIZE = "m"            # "s" | "m" | "l"
PROJECT    = "runs/fomo"
RUN_NAME   = f"sjsu_headcount_all_scenes_{MODEL_SIZE}"

model = FOMO(model_path=None, size=MODEL_SIZE, nb_classes=1, device=device)
print(f"Model: FOMO-{MODEL_SIZE}")
print(f"  Input size : {model.input_size}×{model.input_size}")
print(f"  Grid size  : {model.input_size//8}×{model.input_size//8}")
print(f"  Classes    : {model.nb_classes}")

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

results = model.train(
    allow_experimental=True,
    data=str(data_yaml_path),
    epochs=EPOCHS,
    batch=BATCH,
    lr0=3e-4,
    eval_interval=1,
    workers=2,
    device=device,
    project=PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    
    # Early stopping
    patience=12,
    
    # Regularization & Speedups
    optimizer="adamw",
    weight_decay=1e-4,
    ema=True,
    amp=True,
    
    # Data Augmentations
    mosaic_prob=1.0,
    flip_prob=0.5,
    hsv_prob=1.0,
)

save_dir = Path(results["save_dir"])
print(f"\nTraining complete. Saved to: {save_dir}")


In [ ]:

from fomo import FOMO

weights_dir = save_dir / "weights"
best_pt  = weights_dir / "best.pt"
last_pt  = weights_dir / "last.pt"
ckpt_path = best_pt if best_pt.exists() else last_pt

trained = FOMO(str(ckpt_path), device=device)
print(f"Loaded: {ckpt_path.name}")
print(f"  family : {trained.family}")
print(f"  size   : {trained.size}")
print(f"  imgsz  : {trained.input_size}")
print(f"  nc     : {trained.nb_classes}")


## 4 — Visualize Model Predictions and Heatmaps

Below, we define a helper function to run the trained model on sample validation images, extract the raw activation values (logits), convert them to probability heatmaps, and overlay them alongside the point predictions.

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

def visualize_fomo_predictions(model, image_path, conf_threshold=0.25):
    # 1. Load original image
    img = Image.open(image_path).convert("RGB")
    w, h = img.size
    
    # 2. Get point predictions using model.predict
    results = model.predict(image_path, conf=conf_threshold)
    
    # 3. Get raw logits/heatmaps by running preprocess + forward pass
    input_tensor, _, _, _ = model._preprocess(image_path)
    with torch.no_grad():
        logits = model._forward(input_tensor.to(model.device))
    
    # 4. Compute probabilities (heatmap)
    # logits shape: (1, C + 1, H_grid, W_grid) where C + 1 is background + classes
    probs = F.softmax(logits, dim=1).cpu().numpy()[0]
    
    # Plotting
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Col 1: Original Image
    axes[0].imshow(img)
    axes[0].set_title("Original Image", fontsize=14)
    axes[0].axis("off")
    
    # Col 2: Heatmap Overlay
    axes[1].imshow(img)
    # Take the maximum class probability heatmap across all foreground classes
    heatmap = probs[1:].max(axis=0)
    # Resize heatmap to original image dimensions for overlay
    heatmap_resized = Image.fromarray((heatmap * 255).astype(np.uint8)).resize((w, h), Image.Resampling.BILINEAR)
    im = axes[1].imshow(heatmap_resized, alpha=0.6, cmap="jet")
    axes[1].set_title("Learned Heatmap Overlay", fontsize=14)
    axes[1].axis("off")
    fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
    
    # Col 3: Point Predictions Overlay
    axes[2].imshow(img)
    if results.points is not None and len(results.points) > 0:
        pts = results.points.xy.cpu().numpy() if isinstance(results.points.xy, torch.Tensor) else results.points.xy
        confs = results.points.conf.cpu().numpy() if isinstance(results.points.conf, torch.Tensor) else results.points.conf
        axes[2].scatter(pts[:, 0], pts[:, 1], color="red", marker="o", s=60, edgecolors="white", linewidths=1.5, label="Predictions")
        for pt, conf in zip(pts, confs):
            axes[2].annotate(f"{conf:.2f}", (pt[0] + 5, pt[1] - 5), color="yellow", fontsize=9, weight="bold",
                             bbox=dict(boxstyle="round,pad=0.2", fc="black", alpha=0.5, ec="none"))
        axes[2].legend(loc="upper right")
    axes[2].set_title("Point Predictions Overlay", fontsize=14)
    axes[2].axis("off")
    
    plt.tight_layout()
    plt.show()

# Run visualization on a sample image from the validation set
val_images = list((DATASET_ROOT / "valid" / "images").glob("*.jpg"))
if val_images:
    visualize_fomo_predictions(trained, val_images[0], conf_threshold=0.25)
else:
    print("No validation images found to visualize.")